# Institution Resolver v3 — 280k Batch Colab Pro Master Notebook

Bu notebook Colab Pro (A100/L4 GPU) uzerinde 280.000 satirlik devasa kurum eslestirme batch'ini en yuksek hizda ve sorunsuz tamamlamak icin hazirlanmistir.

### Uygulanan Optimizasyonlar:
1. **Yerel NVMe SSD:** Ollama model agirliklari Google Drive'dan yerel `/content/` diskine kopyalanir. Drive FUSE gecikmesi ortadan kalkar.
2. **KEEP_ALIVE=-1:** Model GPU VRAM'den asla dusurulmez. Cold-start timeout'lari sifira iner.
3. **Model On-Isitma (Warm-up):** Batch baslamadan once model VRAM'a yuklenip test edilir.
4. **torchvision/torchaudio Temizligi:** Colab CUDA surum cakismalari pip uninstall ile onlenir.
5. **Git Dali:** Klonlama guncel `feat/gate-asama1` dali uzerinden yapilir.
6. **Yerel SSD Cikti:** Ciktilar `/content/` uzerine yazilir, islem bitince Drive'a kopyalanir.
7. **Resume:** `--resume` bayragi sayesinde kopma/yeniden baslatmada kaldiginiz satirdan devam eder.

## 1) Donanim & GPU Kontrolu

In [ ]:
!nvidia-smi

## 2) Google Drive Baglama + Dizin Yollari

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_ROOT = "/content/drive/MyDrive/institution_resolver_v3"
DRIVE_RAW = f"{DRIVE_ROOT}/data_raw"
DRIVE_PROCESSED = f"{DRIVE_ROOT}/data_processed"
DRIVE_JOBS = f"{DRIVE_ROOT}/jobs"
DRIVE_EVAL = f"{DRIVE_ROOT}/data_eval"
DRIVE_OLLAMA = f"{DRIVE_ROOT}/ollama_models"
DRIVE_OUTPUT = f"{DRIVE_ROOT}/output"

for p in (DRIVE_RAW, DRIVE_PROCESSED, DRIVE_JOBS, DRIVE_EVAL, DRIVE_OLLAMA, DRIVE_OUTPUT):
    os.makedirs(p, exist_ok=True)

print("Drive klasorleri hazir:", DRIVE_ROOT)

## 3) Koda Erisim (Git Clone / Pull)

In [ ]:
REPO_DIR = "/content/institution_resolver_v3"
BRANCH = "feat/gate-asama1"

import os
if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} https://github.com/mcangultekin/institution_resolver_v3.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull
%cd {REPO_DIR}

## 4) Drive Sembolik Linkleri

In [ ]:
import os, shutil

def _link(name, target):
    os.makedirs(target, exist_ok=True)
    link = f"data/{name}"
    if os.path.islink(link):
        return
    if os.path.isdir(link):
        for f in os.listdir(link):
            src, dst = f"{link}/{f}", f"{target}/{f}"
            if not os.path.exists(dst):
                shutil.move(src, dst)
        shutil.rmtree(link)
    os.symlink(target, link)

os.makedirs("data", exist_ok=True)
_link("raw", DRIVE_RAW)
_link("processed", DRIVE_PROCESSED)
_link("jobs", DRIVE_JOBS)

!ls -la data

## 5) Elasticsearch Kurulumu ve Baslatma

In [ ]:
%%bash
set -e
ES_VERSION=8.14.0
if [ ! -d /content/es ]; then
  wget -q https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-${ES_VERSION}-linux-x86_64.tar.gz -O /content/es.tar.gz
  mkdir -p /content/es
  tar -xzf /content/es.tar.gz -C /content/es --strip-components=1
fi

grep -q '^discovery.type' /content/es/config/elasticsearch.yml || cat >> /content/es/config/elasticsearch.yml <<EOF
discovery.type: single-node
xpack.security.enabled: false
xpack.security.http.ssl.enabled: false
xpack.ml.enabled: false
EOF

cat > /content/es/config/elasticsearch.policy <<'POLEOF'
grant {
  permission java.io.FilePermission "/sys/-", "read";
  permission java.io.FilePermission "/proc/-", "read";
};
POLEOF

sysctl -w vm.max_map_count=262144 || true
id -u esuser &>/dev/null || useradd -m esuser
chown -R esuser:esuser /content/es

pkill -f 'org.elasticsearch.bootstrap.Elasticsearch' 2>/dev/null || true
sleep 1
sudo -u esuser env ES_JAVA_OPTS="-Xms4g -Xmx4g -Djava.security.policy=/content/es/config/elasticsearch.policy" \
  setsid /content/es/bin/elasticsearch < /dev/null > /content/es/es.log 2>&1 &
disown
sleep 3

In [ ]:
import time, requests

for _ in range(60):
    try:
        r = requests.get("http://localhost:9200/_cluster/health", timeout=2)
        if r.status_code == 200:
            print("Elasticsearch Saglikli:", r.json())
            break
    except Exception:
        pass
    time.sleep(2)
else:
    raise RuntimeError("ES baslatilamadi, /content/es/es.log kontrol edin")

## 6) Ollama Kurulumu (Yerel SSD + KEEP_ALIVE Optimizasyonu)

Model agirliklari Google Drive'dan yerel NVMe SSD'ye kopyalanir.
`OLLAMA_KEEP_ALIVE=-1` ile model GPU VRAM'den asla dusurulmez.
Batch baslamadan once model bir test sorgusuyla on-isitilir (warm-up).

In [ ]:
%%bash
set -e
apt-get update -qq && apt-get install -y -qq zstd
command -v ollama &> /dev/null || curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import os, subprocess, time, requests, shutil

# Model dosyalarini Drive'dan yerel SSD'ye kopyala
LOCAL_OLLAMA = "/content/ollama_local"

if not os.path.isdir(LOCAL_OLLAMA):
    print("Model dosyalari yerel SSD'ye kopyalaniyor (1-2 dk)...")
    shutil.copytree(DRIVE_OLLAMA, LOCAL_OLLAMA)
    print("Kopyalama tamamlandi!")
else:
    print("Model zaten yerel SSD de mevcut.")

# Ollama ortam degiskenleri
os.environ["OLLAMA_MODELS"] = LOCAL_OLLAMA
os.environ["OLLAMA_NUM_PARALLEL"] = "8"
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"  # ASLA VRAM den dusurme

# Eski sureci kapat, yenisini baslat
subprocess.run(["pkill", "-f", "ollama"], capture_output=True)
time.sleep(2)

subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/content/ollama.log", "a"),
    stderr=subprocess.STDOUT,
    env=os.environ,
)

for _ in range(30):
    try:
        if requests.get("http://localhost:11434/api/tags", timeout=2).status_code == 200:
            print("Ollama servisi baslatildi (yerel SSD + KEEP_ALIVE=-1)")
            break
    except Exception:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Ollama baslatilamadi")

# Gemma 4 E4B modelini indir (yoksa)
!ollama pull gemma4:e4b

# Modeli VRAM a on-yukle (warm-up)
print("Model VRAM a on-yukleniyor (warm-up)...")
r = requests.post("http://localhost:11434/api/generate", json={
    "model": "gemma4:e4b",
    "prompt": "Merhaba",
    "stream": False,
})
if r.status_code == 200:
    print("Model GPU belleginde sicak ve hazir!")
else:
    print("UYARI: Warm-up basarisiz, log kontrol edin:", r.text[:200])

## 7) Python Paket Kurulumu + torchvision/torchaudio Cakisma Temizligi

In [ ]:
# Colab daki PyTorch CUDA surum cakismasini onle:
!pip uninstall -y torchaudio torchvision 2>/dev/null || true

# Projeyi kur:
!pip install -q --force-reinstall -e ".[dev,embed,llm,api]"

## 8) Elasticsearch Sema Sifirlama ve Indeksleme

In [ ]:
# ES 400 BadRequestError hatasini onlemek icin setup-es sifirlanir:
!python3 -m institution_resolver_v3.cli.main setup-es
!python3 -m institution_resolver_v3.cli.main index --embeddings

## 9) Tekli Sorgu Testi (Her Sey Calisiyor mu?)

Batch'e baslamadan once tek bir sorguyla tum pipeline'in sorunsuz calistigini dogrulayalim.

In [ ]:
!python3 -m institution_resolver_v3.cli.main judge "gazi universitesi istatistik bolumu" --model "gemma4:e4b"

## 10) Opsiyonel: Worker Hiz Testi (Benchmark)

100 ornek sorgu uzerinde farkli worker sayilarini hiz acisindan test eder.

In [ ]:
%%writefile /content/benchmark.py
import sys, os, time, csv
sys.path.insert(0, "/content/institution_resolver_v3/src")

from institution_resolver_v3.eval.csv_runner import run_csv_batch
from institution_resolver_v3.eval.gate_batch import process_one_gate, FIELDNAMES

queries = []
INPUT_CSV = "/content/drive/MyDrive/institution_resolver_v3/jobs/batch_input_parent_empty.csv"
with open(INPUT_CSV, encoding="utf-8") as f:
    for row in csv.DictReader(f):
        q = (row.get("query") or "").strip()
        if q:
            queries.append(q)
        if len(queries) >= 100:
            break

print("WORKER HIZ TESTI BASLIYOR (100 Ornek Sorgu)...")

for workers in [1, 2, 4, 8, 12, 16]:
    out_tmp = f"/content/test_w{workers}.csv"
    t0 = time.time()

    def _proc(q):
        return process_one_gate(q, top=5)

    run_csv_batch(queries, out_tmp, FIELDNAMES, _proc, limit=100, resume=False, max_workers=workers)
    dt = time.time() - t0
    qps = len(queries) / dt
    print(f"Workers = {workers:>2d} -> Sure: {dt:.2f} sn | Hiz: {qps:.1f} sorgu/saniye")

print("Test tamamlandi!")

In [ ]:
!python3 /content/benchmark.py

# --- BATCH CALISTIRMA SECENEKLERI ---

Ihtiyaciniza uygun olan asagidaki 3 calistirma seceneginden birini secip ilgili hucreyi calistiriniz.

* Tum seceneklerde ciktilar Colab yerel SSD diski (`/content/`) uzerinde maksimum hizda islenir, islem bitince otomatik Drive'a yedeklenir.
* `--resume` sayesinde kopma/restart durumunda kaldiginiz satirdan devam eder.

### SECENEK 1: Gate + LLM Hibrit Batch (decide-batch + Gemma 4 E4B)
* **Aciklama:** Gate'in kararsiz kaldigi satirlar Gemma 4 E4B modeline gider. En yuksek kaliteli ciktiyi uretir.
* **Model:** `gemma4:e4b` (yerel SSD + KEEP_ALIVE=-1 ile optimize)
* **Tahmini Sure:** ~5-7 saat (yerel SSD optimizasyonu ile)

In [ ]:
import os, shutil

INPUT_CSV = f"{DRIVE_JOBS}/batch_input_parent_empty.csv"
LOCAL_OUTPUT = "/content/280k_decide_batch_sonuc.csv"
DRIVE_FINAL_OUTPUT = f"{DRIVE_OUTPUT}/280k_decide_batch_sonuc.csv"
MODEL_TAG = "gemma4:e4b"

# Varsa onceden islenen kismi yerel diske kopyala (resume icin):
if os.path.exists(DRIVE_FINAL_OUTPUT) and not os.path.exists(LOCAL_OUTPUT):
    shutil.copy(DRIVE_FINAL_OUTPUT, LOCAL_OUTPUT)

print("SECENEK 1: Gate + Gemma 4 E4B (decide-batch) Basliyor...")

!python3 -m institution_resolver_v3.cli.main decide-batch "{INPUT_CSV}" --query-col query --model "{MODEL_TAG}" --out "{LOCAL_OUTPUT}" --workers 8 --resume

# Islem tamamlaninca otomatik Drive a yedekle:
shutil.copy(LOCAL_OUTPUT, DRIVE_FINAL_OUTPUT)
print("ISLEM BITTI! Sonuc Google Drive a kopyalandi:", DRIVE_FINAL_OUTPUT)

### SECENEK 2: Standart Gate Batch (LLM Yok)
* **Aciklama:** LLM cagirmaz. Tum 280.000 satiri sadece Elasticsearch + Deterministik Gate ile isler.
* **Tahmini Sure:** ~3.5-4 saat

In [ ]:
import os, shutil

INPUT_CSV = f"{DRIVE_JOBS}/batch_input_parent_empty.csv"
LOCAL_OUTPUT = "/content/280k_gate_batch_sonuc.csv"
DRIVE_FINAL_OUTPUT = f"{DRIVE_OUTPUT}/280k_gate_batch_sonuc.csv"

if os.path.exists(DRIVE_FINAL_OUTPUT) and not os.path.exists(LOCAL_OUTPUT):
    shutil.copy(DRIVE_FINAL_OUTPUT, LOCAL_OUTPUT)

print("SECENEK 2: Standart Gate-Batch (LLM Yok) Basliyor...")

!python3 -m institution_resolver_v3.cli.main gate-batch "{INPUT_CSV}" --query-col query --out "{LOCAL_OUTPUT}" --resume

shutil.copy(LOCAL_OUTPUT, DRIVE_FINAL_OUTPUT)
print("ISLEM BITTI! Sonuc Google Drive a kopyalandi:", DRIVE_FINAL_OUTPUT)

### SECENEK 3: Envanter Gate Batch (inventory-batch --no-judge)
* **Aciklama:** Envanter verileri icin ozel mod. LLM kapali (`--no-judge`). Karara girmeyen top-1 adaylari da saklar.
* **Tahmini Sure:** ~4-5 saat

In [ ]:
import os, shutil

INPUT_CSV = f"{DRIVE_JOBS}/batch_input_parent_empty.csv"
LOCAL_OUTPUT = "/content/280k_inventory_gate_sonuc.csv"
DRIVE_FINAL_OUTPUT = f"{DRIVE_OUTPUT}/280k_inventory_gate_sonuc.csv"

if os.path.exists(DRIVE_FINAL_OUTPUT) and not os.path.exists(LOCAL_OUTPUT):
    shutil.copy(DRIVE_FINAL_OUTPUT, LOCAL_OUTPUT)

print("SECENEK 3: Envanter Gate-Batch (inventory-batch --no-judge) Basliyor...")

!python3 -m institution_resolver_v3.cli.main inventory-batch "{INPUT_CSV}" --query-col query --no-judge --out "{LOCAL_OUTPUT}" --workers 8 --resume

shutil.copy(LOCAL_OUTPUT, DRIVE_FINAL_OUTPUT)
print("ISLEM BITTI! Sonuc Google Drive a kopyalandi:", DRIVE_FINAL_OUTPUT)